# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q emoji wordcloud
    print('Packages installed.')


# WhatsApp Chat Analysis

This notebook demonstrates how to parse, clean, and analyze exported WhatsApp chat data. We will use regular expressions to extract dates, times, senders, and messages, and then use pandas and matplotlib/seaborn for analysis and visualization.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from datetime import datetime
import emoji
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style("whitegrid")


## 1. Data Loading and Parsing

WhatsApp exports are typically text files where each line (or block) represents a message. The format varies by device (iOS vs Android). We'll define a function to parse standard formats.

In [ ]:
def parse_whatsapp_chat(filepath):
    '''
    Parses a WhatsApp chat export file (txt) into a Pandas DataFrame.
    Handles both Android (dd/mm/yyyy, hh:mm - ) and iOS ([dd/mm/yyyy, hh:mm:ss] ) formats.
    '''
    with open(filepath, 'r', encoding='utf-8') as f:
        data = f.readlines()
        
    messages = []
    # Regular expressions for common formats
    # Android: 12/04/2023, 14:30 - User: Message
    android_regex = r'^(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}) - (.*?): (.*)$'
    # iOS: [12/04/2023, 14:30:15] User: Message
    ios_regex = r'^\[(\d{1,2}/\d{1,2}/\d{2,4}), (\d{1,2}:\d{2}:\d{2})\] (.*?): (.*)$'
    
    for line in data:
        line = line.strip()
        if not line:
            continue
            
        # Try Android format
        match = re.match(android_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Try iOS format
        match = re.match(ios_regex, line)
        if match:
            date, time, sender, message = match.groups()
            messages.append([date, time, sender, message])
            continue
            
        # Handle multi-line messages (append to previous message)
        if messages:
            messages[-1][-1] += " " + line

    df = pd.DataFrame(messages, columns=['Date', 'Time', 'Sender', 'Message'])
    
    # Convert Date and Time to datetime objects
    # Note: Format might need adjustment depending on locale
    try:
        df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M')
    except ValueError:
        # Try with seconds for iOS
        try:
             df['datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%y %H:%M:%S')
        except:
             # Fallback or more complex parsing if needed
             pass
    
    return df

# Example usage (Uncomment and replace with your file path)
# df = parse_whatsapp_chat('your_chat_export.txt')

# Creating dummy data for demonstration
dummy_data = [
    "12/01/2023, 10:00 - Alice: Hey everyone! How are we doing?",
    "12/01/2023, 10:05 - Bob: Good morning Alice! I am doing great.",
    "12/01/2023, 10:06 - Charlie: Hi guys. I'm running a bit late.",
    "12/01/2023, 10:10 - Alice: No worries Charlie. We haven't started yet.",
    "13/01/2023, 09:00 - Bob: Did anyone see the news?",
    "13/01/2023, 09:15 - Alice: Yes, it is crazy! 😱",
    "13/01/2023, 09:20 - Charlie: I missed it. What happened?",
    "14/01/2023, 18:00 - Alice: Meeting at 5pm tomorrow?",
    "14/01/2023, 18:05 - Bob: Works for me.",
    "14/01/2023, 18:10 - Charlie: 👍"
]

with open('dummy_chat.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(dummy_data))

df = parse_whatsapp_chat('dummy_chat.txt')
print(df.head())


## 2. Basic Statistics

Let's calculate some basic statistics like total messages, active users, and messages per day.

In [ ]:
# Total messages
print(f"Total Messages: {len(df)}")

# Active users
print(f"Active Users: {df['Sender'].unique()}")

# Messages per user
user_counts = df['Sender'].value_counts()
print("\nMessages per User:\n", user_counts)

# Plot messages per user
plt.figure(figsize=(8, 5))
sns.barplot(x=user_counts.index, y=user_counts.values, palette='viridis')
plt.title('Total Messages sent by each User')
plt.ylabel('Number of Messages')
plt.show()


## 3. Temporal Analysis

Analyzing activity over time (e.g., messages per day, hour of day).

In [ ]:
if 'datetime' in df.columns:
    df['Date_only'] = df['datetime'].dt.date
    df['Hour'] = df['datetime'].dt.hour
    
    # Daily activity
    daily_counts = df.groupby('Date_only').count()['Message']
    
    plt.figure(figsize=(10, 5))
    daily_counts.plot(kind='line', marker='o')
    plt.title('Daily Message Frequency')
    plt.ylabel('Count')
    plt.show()
    
    # Hourly activity
    hourly_counts = df['Hour'].value_counts().sort_index()
    
    plt.figure(figsize=(10, 5))
    sns.barplot(x=hourly_counts.index, y=hourly_counts.values, color='skyblue')
    plt.title('Activity by Hour of Day')
    plt.xlabel('Hour (24h)')
    plt.ylabel('Message Count')
    plt.show()


## 4. Content Analysis

Let's analyze the content, such as finding the most common words or emojis.

In [ ]:
def extract_emojis(text):
    return [c for c in text if c in emoji.EMOJI_DATA]

df['Emojis'] = df['Message'].apply(extract_emojis)
all_emojis = [e for sublist in df['Emojis'] for e in sublist]
emoji_counts = Counter(all_emojis)

if emoji_counts:
    print("Top 5 Emojis:", emoji_counts.most_common(5))
else:
    print("No emojis found.")

# Word Cloud
from wordcloud import WordCloud

text = " ".join(message for message in df.Message)

# Simple cleaning
text_cleaned = re.sub(r'http\S+', '', text) # remove urls
text_cleaned = re.sub(r'[^a-zA-Z\s]', '', text_cleaned) # remove special chars (simplistic)

wordcloud = WordCloud(width=800, height=400, background_color='white').generate(text_cleaned)

plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Chat Messages')
plt.show()
